In [1]:
# 1. Cài đặt thư viện cần thiết
!pip install -q diffusers transformers accelerate peft opencv-python

# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✅ Đã cài đặt và kết nối Drive thành công!")

Mounted at /content/drive
✅ Đã cài đặt và kết nối Drive thành công!


In [2]:
import torch
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from io import BytesIO
import ipywidgets as widgets
from IPython.display import display, clear_output

from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
from peft import PeftModel

# ================= CẤU HÌNH ĐƯỜNG DẪN DRIVE =================
LORA_PATH = "/content/drive/MyDrive/NguyenGiaTri/nguyengiatri_model"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"⏳ Đang tải model trên {DEVICE} (Mất khoảng 30s)...")

# ================= LOAD MODEL =================
try:
    # 1. Load ControlNet
    controlnet = ControlNetModel.from_pretrained(
        "lllyasviel/sd-controlnet-canny", torch_dtype=torch.float16
    )

    # 2. Load Base Model
    pipe = StableDiffusionControlNetPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        controlnet=controlnet,
        torch_dtype=torch.float16,
        safety_checker=None
    ).to(DEVICE)

    pipe.scheduler = UniPCMultistepScheduler.from_pretrained(
        "runwayml/stable-diffusion-v1-5", subfolder="scheduler"
    )

    # 3. Load LoRA từ Drive & Merge
    pipe.unet = PeftModel.from_pretrained(pipe.unet, LORA_PATH)
    pipe.unet = pipe.unet.merge_and_unload()

    print(f"✅ Đã load thành công Model từ: {LORA_PATH}")

except Exception as e:
    print(f"❌ Lỗi: {e}")
    print("👉 Hãy kiểm tra lại đường dẫn folder trong Google Drive xem có đúng tên không.")

# ================= HÀM XỬ LÝ ẢNH & VISUALIZE =================

def get_canny_image(pil_image):
    """Tạo ảnh nét (Canny)"""
    w, h = pil_image.size
    # Resize về bội số của 8 và giới hạn size để chạy nhanh
    max_size = 768
    scale = max_size / max(w, h)
    new_w, new_h = int(w * scale), int(h * scale)
    new_w, new_h = (new_w // 8) * 8, (new_h // 8) * 8

    resized_image = pil_image.resize((new_w, new_h))

    img_cv = np.array(resized_image)
    img_cv = cv2.Canny(img_cv, 100, 200)
    img_cv = img_cv[:, :, None]
    img_cv = np.concatenate([img_cv, img_cv, img_cv], axis=2)
    return Image.fromarray(img_cv), resized_image

def latents_to_rgb(pipe, latents):
    """Giải mã nhiễu (Latent) sang ảnh màu (RGB)"""
    latents = 1 / 0.18215 * latents
    with torch.no_grad():
        image = pipe.vae.decode(latents).sample
    image = (image / 2 + 0.5).clamp(0, 1)
    image = image.cpu().permute(0, 2, 3, 1).float().numpy()
    return Image.fromarray((image[0] * 255).round().astype("uint8"))

# List chứa ảnh các bước
intermediate_images = []

def callback_fn(pipe, step_index, timestep, callback_kwargs):
    latents = callback_kwargs["latents"]
    # Các bước muốn chụp lại ảnh
    capture_steps = [0, 5, 10, 15, 20, 25]

    if step_index in capture_steps or step_index == 29:
        img = latents_to_rgb(pipe, latents)
        intermediate_images.append((step_index, img))
    return callback_kwargs

# ================= GIAO DIỆN UPLOAD =================

uploader = widgets.FileUpload(
    accept='image/*', multiple=False, description='Tải ảnh lên'
)
btn_run = widgets.Button(
    description='▶️ CHẠY & VẼ QUÁ TRÌNH',
    button_style='warning',
    layout=widgets.Layout(width='300px')
)
out_display = widgets.Output()

display(widgets.VBox([uploader, btn_run]))
display(out_display)

def on_click_run(b):
    with out_display:
        clear_output()
        if not uploader.value:
            print("⚠️ Bạn chưa chọn ảnh!")
            return

        # Đọc file upload
        uploaded_filename = list(uploader.value.keys())[0]
        content = uploader.value[uploaded_filename]['content']
        init_image = Image.open(BytesIO(content)).convert("RGB")

        print("🎨 Đang xử lý... (Vui lòng đợi 10-20 giây)")

        # 1. Tạo Canny Map
        canny_control, resized_init = get_canny_image(init_image)

        # 2. Reset list ảnh
        global intermediate_images
        intermediate_images = []

        # 3. Cấu hình Prompt
        prompt = "nguyen gia tri style, vietnamese lacquer painting, gold leaf texture, eggshell mosaic, warm tones, masterpiece, highly detailed"
        n_prompt = "low quality, bad anatomy, blurry, 3d, realistic photo, black and white"

        # 4. Chạy AI
        generator = torch.Generator(device=DEVICE).manual_seed(42)

        result = pipe(
            prompt,
            image=canny_control,
            negative_prompt=n_prompt,
            num_inference_steps=30,
            guidance_scale=9.0,
            controlnet_conditioning_scale=0.8,
            generator=generator,
            callback_on_step_end=callback_fn # Gọi hàm chụp ảnh
        ).images[0]

        print("✅ Xong! Dưới đây là quá trình biến đổi từ nhiễu sang tranh:")

        # 5. Vẽ biểu đồ
        cols = len(intermediate_images) + 1
        fig, axes = plt.subplots(1, cols, figsize=(24, 5))

        # Ảnh gốc
        axes[0].imshow(resized_init)
        axes[0].set_title("Ảnh Gốc")
        axes[0].axis("off")

        # Các bước biến đổi
        for i, (step, img) in enumerate(intermediate_images):
            ax = axes[i+1]
            ax.imshow(img)
            if step == 0:
                ax.set_title(f"Bước {step}\n(Nhiễu khởi tạo)")
            elif step == 29:
                ax.set_title(f"Kết quả\n(Sơn mài)")
            else:
                ax.set_title(f"Bước {step}/30")
            ax.axis("off")

        plt.tight_layout()
        plt.show()

btn_run.on_click(on_click_run)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


⏳ Đang tải model trên cuda (Mất khoảng 30s)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/920 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/1.45G [00:00<?, ?B/s]

...

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!
You have disabled the safety checker for <class 'diffusers.pipelines.controlnet.pipeline_controlnet.StableDiffusionControlNetPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


✅ Đã load thành công Model từ: /content/drive/MyDrive/NguyenGiaTri/nguyengiatri_model


Output()